In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Klasický feedforward a řízení toku (dynamické obvody)

<Accordion>
<AccordionItem title="Verze balíčků">

Kód na této stránce byl vyvinut s použitím následujících závislostí.
Doporučujeme používat tyto nebo novější verze.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
Dynamické obvody jsou mocné nástroje, pomocí nichž lze měřit Qubity uprostřed provádění kvantového Circuit a poté v rámci Circuit provádět operace klasické logiky na základě výsledků těchto průběžných měření. Tento postup se také nazývá _klasický feedforward_. Přestože je porozumění tomu, jak nejlépe využít dynamické obvody, teprve v počátcích, komunita kvantového výzkumu již identifikovala řadu případů použití, například:

* Efektivní příprava kvantových stavů, například [GHZ stav](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [W-stav](https://arxiv.org/abs/2403.07604) (další informace o W-stavu najdeš také v ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), a rozsáhlá třída [maticových produktových stavů](https://arxiv.org/abs/2404.16083)
* [Efektivní provázání na velké vzdálenosti](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) mezi Qubity na stejném čipu pomocí mělkých obvodů
* Efektivní [vzorkování obvodů podobných IQP](https://arxiv.org/pdf/2505.04705)

Qiskit podporuje čtyři konstrukty řízení toku pro klasický feedforward, každý implementovaný jako metoda na [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit). Konstrukty a jejich odpovídající metody jsou:

- Příkaz if - [`QuantumCircuit.if_test`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test)
- Příkaz switch - [`QuantumCircuit.switch`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#switch)
- Smyčka for - [`QuantumCircuit.for_loop`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#for_loop)
- Smyčka while - [`QuantumCircuit.while_loop`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#while_loop)

Každá z těchto metod vrací [správce kontextu](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) a obvykle se používá v příkazu `with`. Zbytek tohoto průvodce vysvětluje každý z těchto konstruktů a jak je používat.

> **Caution:** Existují určitá omezení operací klasického feedforwardu a řízení toku na kvantovém hardwaru, která mohou ovlivnit tvůj program. Další informace najdeš v části [Spuštění dynamických obvodů](/guides/execute-dynamic-circuits).

## Příkaz `if`
Příkaz `if` se používá k podmíněnému provádění operací na základě hodnoty klasického bitu nebo registru.

V níže uvedeném příkladu aplikujeme Hadamardovu Gate na qubit a změříme jej. Pokud je výsledek 1, aplikujeme na qubit Gate X, čímž jej přepneme zpět do stavu 0. Poté qubit znovu změříme. Výsledný výsledek měření by měl být 0 se 100% pravděpodobností.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif)

Příkazu `with` lze přiřadit cíl, který je sám správcem kontextu a lze jej uložit a následně použít k vytvoření bloku else, jenž se provede vždy, když se obsah bloku `if` *neprovede*.

V níže uvedeném příkladu inicializujeme registry se dvěma Qubity a dvěma klasickými bity. Na první qubit aplikujeme Hadamardovu Gate a změříme jej. Pokud je výsledek 1, aplikujeme Hadamardovu Gate na druhý qubit; v opačném případě aplikujeme Gate X na druhý qubit. Nakonec změříme i druhý qubit.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif)

Kromě podmíněnosti na jednom klasickém bitu je také možné podmíněnost vztáhnout na hodnotu klasického registru složeného z více bitů.

V níže uvedeném příkladu aplikujeme Hadamardovy Gate na dva Qubity a změříme je. Pokud je výsledek `01`, tedy první qubit je 1 a druhý qubit je 0, aplikujeme Gate X na třetí qubit. Nakonec změříme třetí qubit. Všimni si, že pro přehlednost jsme se rozhodli specifikovat stav třetího klasického bitu, který je 0, v podmínce `if`. Ve schématu Circuit je podmínka znázorněna kruhy na klasických bitech, na které se podmínka vztahuje. Plný kruh označuje podmíněnost na 1, zatímco prázdný kruh označuje podmíněnost na 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif)

## Příkaz switch
Příkaz switch se používá k výběru akcí na základě hodnoty klasického bitu nebo registru. Je podobný příkazu if, ale umožňuje zadat více případů pro logiku větvení. Níže uvedený příklad aplikuje Hadamardovu Gate na qubit a změří jej. Pokud je výsledek 0, aplikuje Gate X na qubit, a pokud je výsledek 1, aplikuje Gate Z. Výsledný výsledek měření by měl být 1 se 100% pravděpodobností.

In [4]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.switch(c0) as case:
    with case(0):
        circuit.x(q0)
    with case(1):
        circuit.z(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif)

Protože výše uvedený příklad používal jeden klasický bit, existovaly pouze dva možné případy, takže stejného výsledku bylo možné dosáhnout pomocí příkazu if-else. Příkaz switch je hlavně užitečný při větvení na základě hodnoty klasického registru složeného z více bitů. Následující příklad ukazuje, jak sestavit výchozí případ, který se provede, pokud žádný z předchozích případů neodpovídá. Všimni si, že v příkazu switch se vždy provede pouze jeden z bloků. Neexistuje žádný fallthrough.

Níže uvedený příklad aplikuje Hadamardovy Gate na dva Qubity a změří je. Pokud je výsledek 00 nebo 11, aplikuje Gate Z na třetí qubit. Pokud je výsledek 01, aplikuje Gate Y. Pokud žádný z předchozích případů neodpovídal, aplikuje Gate X. Nakonec změří třetí qubit.

In [5]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.switch(clbits) as case:
    with case(0b000, 0b011):
        circuit.z(q2)
    with case(0b001):
        circuit.y(q2)
    with case(case.DEFAULT):
        circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 267, '110': 249, '011': 265, '000': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif)

## Smyčka for
Smyčka for se používá k iteraci přes posloupnost klasických hodnot a provádění určitých operací během každé iterace.

Následující příklad používá smyčku for k aplikaci 5 Gate X na qubit a poté jej změří. Protože provádí lichý počet Gate X, celkový efekt je přepnutí Qubitu ze stavu 0 do stavu 1.

In [6]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

with circuit.for_loop(range(5)) as _:
    circuit.x(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif)

## Smyčka while
Smyčka while se používá k opakování instrukcí, dokud je splněna určitá podmínka.

Níže uvedený příklad aplikuje Hadamardovy Gate na dva Qubity a změří je. Poté vytvoří smyčku while, která tento postup opakuje, dokud je výsledek měření 11. Výsledné finální měření by tedy nikdy nemělo být 11, přičemž zbývající možnosti se vyskytují přibližně se stejnou frekvencí.

In [7]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)

q0, q1 = qubits
c0, c1 = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.while_loop((clbits, 0b11)):
    circuit.h([q0, q1])
    circuit.measure(q0, c0)
    circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 334, '10': 368, '00': 322}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif)

## Klasické výrazy
Modul klasických výrazů Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) obsahuje experimentální reprezentaci runtime operací na klasických hodnotách během provádění Circuit.

Následující příklad ukazuje, že výpočet parity lze použít k vytvoření n-Qubitového GHZ stavu pomocí dynamických obvodů. Nejprve vygeneruj $n/2$ Bellových párů na sousedních Qubitech. Poté tyto páry slepuj dohromady pomocí vrstvy CNOT Gate mezi páry. Následně změř cílový qubit všech předchozích CNOT Gate a každý změřený qubit resetuj do stavu $\vert 0 \rangle$. Aplikuj $X$ na každé neměřené místo, pro které je parita všech předchozích bitů lichá. Nakonec se na změřené Qubity aplikují CNOT Gate, aby se obnovilo provázání ztracené při měření.

Při výpočtu parity první prvek konstruovaného výrazu zahrnuje zvednutí Python objektu `mr[0]` na uzel [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` se používá k převodu libovolných objektů na klasické výrazy). To není nutné pro `mr[1]` a případný následující klasický registr, protože jsou vstupem do `expr.bit_xor` a veškeré potřebné zvednutí se v těchto případech provede automaticky. Takové výrazy lze sestavovat ve smyčkách a dalších konstruktech.

In [8]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [9]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif)

<span id="store"></span>
### `Store`

Instrukci [`store`](https://docs.quantum.ibm.com/api/qiskit/circuit#store) můžeš použít k uložení výsledku klasického výrazu, pokud bude tento výraz použit opakovaně. Operace jsou automaticky paralelizovány, což výrazně zvyšuje efektivitu kódu za běhu.

Například je přirozenější a efektivnější za běhu zapsat $B[0] \oplus B[1] \oplus B[2] \ldots$, kde $B = \neg A$, než $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$.  První varianta vypočítá negaci v jediném paralelním kroku před řetězcem XOR, místo aby každou negaci vyhodnocovala postupně uvnitř výrazu.

Úplný příklad:

In [10]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

qregs = QuantumRegister(4, "q")
creg = ClassicalRegister(3, "c")
# temp is a plain ClassicalRegister used as the store target
temp = ClassicalRegister(3, "temp")
qc = QuantumCircuit(qregs, creg, temp)

qc.h([0, 1, 2])
qc.measure([0, 1, 2], creg)

# Store bit-NOT of the full 3-bit register into temp
qc.store(temp, expr.bit_not(creg))

# Compute parity of temp using bit-indexed XOR
parity = expr.bit_xor(
    expr.bit_xor(expr.index(temp, 0), expr.index(temp, 1)),
    expr.index(temp, 2),
)

# Flip q3 if parity of ~creg is 1
with qc.if_test(parity):
    qc.x(3)

qc.measure([0, 1, 2], creg)

qc.draw("mpl")

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif)

## Další kroky
> **Tip:** - Nauč se implementovat přesné dynamické oddělení pomocí [stretch](/guides/stretch).
> - Použij [vizualizaci časování Circuit](/guides/qiskit-runtime-circuit-timing) k ladění a optimalizaci svých dynamických obvodů.
> - [Spusť dynamické obvody](/guides/execute-dynamic-circuits).